# Silver Layer – Diseases Dataset

## Description
This notebook processes the **diseases dataset** from the Bronze layer and creates a cleaned and standardized **Silver table**.

The goal of the Silver layer is to improve data quality by applying basic transformations such as trimming, normalization, and safe date parsing while preserving the core business attributes.

## Source
Bronze Table:
capstone.bronze.diseases

## Target
Silver Table:
capstone.silver.diseases

## Transformations Applied
The following transformations are performed:

- Trim whitespace from key text fields
- Normalize text casing (disease names, categories)
- Convert chronic flag to boolean
- Extract standardized diagnosis (ICD-like) codes
- Parse raw date fields into DATE format
- Preserve important metadata fields

## Output Schema
The resulting Silver table contains the following columns:

- disease_id
- disease_name
- disease_name_alt
- category
- description
- severity_label
- is_chronic
- diagnosis_code
- treatment_notes_raw
- status_raw
- created_date
- updated_date
- source_system
- source_record_id

## Data Architecture
This table supports downstream joins with:

- medications (via disease_id)
- insurance coverage analysis
- treatment analysis

In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ==================================================
# Setup
# ==================================================
spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# ==================================================
# Read Bronze table
# ==================================================
df = spark.table("capstone.bronze.diseases")

# ==================================================
# Silver transformations
# ==================================================
silver = (
    df
    .withColumn("disease_id", upper(trim(col("disease_id"))))

    .withColumn(
        "disease_name",
        when(trim(col("disease_name")) == "", None)
        .otherwise(initcap(lower(trim(col("disease_name")))))
    )

    .withColumn(
        "disease_name_alt",
        when(trim(col("disease_name_alt")) == "", None)
        .otherwise(initcap(lower(trim(col("disease_name_alt")))))
    )

    .withColumn(
        "category",
        when(trim(col("category")) == "", None)
        .otherwise(
            initcap(
                regexp_replace(
                    regexp_replace(
                        regexp_replace(lower(trim(col("category"))), "[-_/]", " "),
                        "\\s+", " "
                    ),
                    "\\s+related$", ""
                )
            )
        )
    )

    .withColumn(
        "severity_label",
        when(trim(col("severity_label")) == "", None)
        .otherwise(lower(trim(col("severity_label"))))
    )

    .withColumn(
        "description",
        when(trim(col("description")) == "", None)
        .otherwise(trim(col("description")))
    )

    .withColumn(
        "treatment_notes",
        when(trim(col("treatment_notes_raw")) == "", None)
        .otherwise(trim(col("treatment_notes_raw")))
    )

    .withColumn(
        "status",
        when(trim(col("status_raw")) == "", None)
        .otherwise(lower(trim(col("status_raw"))))
    )

    .withColumn(
        "is_chronic",
        when(lower(trim(col("chronic_flag_raw"))).isin("yes", "y", "true", "1", "chronic"), True)
        .when(lower(trim(col("chronic_flag_raw"))).isin("no", "n", "false", "0", "non-chronic"), False)
        .otherwise(None)
    )

    .withColumn(
        "diagnosis_code",
        regexp_extract(
            upper(trim(col("diagnosis_code_raw"))),
            r"([A-Z][0-9]{2,3}(?:\.[0-9A-Z]+)?)",
            1
        )
    )
    .withColumn(
        "diagnosis_code",
        when(col("diagnosis_code") == "", None).otherwise(col("diagnosis_code"))
    )

    # ==================================================
    # Safe date parsing with try_to_timestamp
    # ==================================================
    .withColumn(
        "created_date",
        coalesce(
            to_date(expr("try_to_timestamp(created_date_raw, 'yyyy-MM-dd')")),
            to_date(expr("try_to_timestamp(created_date_raw, 'yyyy/MM/dd')")),
            to_date(expr("try_to_timestamp(created_date_raw, 'MM/dd/yyyy')")),
            to_date(expr("try_to_timestamp(created_date_raw, 'MM-dd-yyyy')")),
            to_date(expr("try_to_timestamp(created_date_raw, 'dd-MMM-yyyy')")),
            to_date(expr("try_to_timestamp(created_date_raw, 'yyyy-MM-dd HH:mm:ss')"))
        )
    )
    .withColumn(
        "updated_date",
        coalesce(
            to_date(expr("try_to_timestamp(updated_date_raw, 'yyyy-MM-dd')")),
            to_date(expr("try_to_timestamp(updated_date_raw, 'yyyy/MM/dd')")),
            to_date(expr("try_to_timestamp(updated_date_raw, 'MM/dd/yyyy')")),
            to_date(expr("try_to_timestamp(updated_date_raw, 'MM-dd-yyyy')")),
            to_date(expr("try_to_timestamp(updated_date_raw, 'dd-MMM-yyyy')")),
            to_date(expr("try_to_timestamp(updated_date_raw, 'yyyy-MM-dd HH:mm:ss')"))
        )
    )
)

# ==================================================
# Deduplication
# ==================================================
window_spec = Window.partitionBy("disease_id").orderBy(
    col("updated_date").desc_nulls_last(),
    col("created_date").desc_nulls_last(),
    col("ingestion_timestamp").desc_nulls_last(),
    col("diagnosis_code").desc_nulls_last(),
    col("disease_name").desc_nulls_last(),
    col("category").desc_nulls_last()
)

silver_dedup = (
    silver
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# ==================================================
# Final columns
# ==================================================
final_df = silver_dedup.select(
    "disease_id",
    "disease_name",
    "disease_name_alt",
    "category",
    "description",
    "severity_label",
    "is_chronic",
    "diagnosis_code",
    "treatment_notes",
    "status",
    "created_date",
    "updated_date",
    "source_system",
    "source_record_id"
)

# ==================================================
# Write table
# ==================================================
spark.sql("DROP TABLE IF EXISTS capstone.silver.diseases")

(
    final_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("capstone.silver.diseases")
)

print("Loaded: capstone.silver.diseases")
print(final_df.dtypes)

In [0]:
%python
display(spark.table("capstone.silver.diseases").limit(20))